In [ ]:
# ============================================================
# Mini-Project: Activity Classification — Task 3 Algorithm
# Nur Kursmethoden: Filter (FIR/IIR), FFT, Autokorrelation,
# Welch Periodogramm, Spectrogram
# ============================================================

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, welch, stft
from DataProcessor import DataProcessor


# =========================
# KONSTANTEN
# =========================
BLOCK_DURATION = 5.0        # Sekunden pro Block (Aufgabenblatt §Task3)
FS = 200                     # Abtastrate Axiamo X2
ACTIVITY_MAP = {
    0: "Nicht klassifizierbar",
    1: "Ruhen",
    2: "Normal Gehen",
    3: "Schnell Gehen",
    4: "Rennen"
}


# =========================
# SCHRITT 1: BANDPASS-FILTER
# Kursreferenz: §3.3 Filter Design [attached_file:1]
# Schritte: Gehaktivitäten liegen typischerweise im Bereich 0.5 – 5 Hz
# =========================
def bandpass_filter(signal, fs, lowcut=0.5, highcut=5.0, order=4):
    """
    FIR-typischer Ansatz: IIR Butterworth Bandpass-Filter.
    Kursreferenz: §3.3 — IIR-Filter mit Butterworth Modell [attached_file:1]
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


# =========================
# SCHRITT 2: AUTO-DMEAN (ACCELERATION TOTAL MAGNITUDE)
# =========================
def compute_total_acc(ax, ay, az):
    """Gesamtbeschleunigung ohne Gravitations-Komponente (mean entfernt)"""
    acc_total = np.sqrt(ax**2 + ay**2 + az**2)
    acc_total -= np.mean(acc_total)
    return acc_total


# =========================
# SCHRITT 3: CADENCE via AUTOSSiON
# Kursreferenz: §2.2 Autokorrelation [attached_file:4]
# Der Peak bei positivem Lag entspricht dem Schrittzyklus
# =========================
def estimate_cadence(acc_total, fs):
    """
    Schätzung der Cadence (Schritte/min) über biased Autokorrelation.
    Kursreferenz: §2.2 — Autokorrelation detektet periodische Strukturen [attached_file:4]
    """
    signal = compute_total_acc(acc_total['x'], acc_total['y'], acc_total['z'])
    
    # Biased Autokorrelation (Kurs: konstante Normierung 1/N)
    corr = np.correlate(signal, signal, mode='full')
    corr = corr / len(signal)  # biased
    lags = np.arange(-len(signal) + 1, len(signal))
    
    # Nur positive lags betrachten (Symmetrie)
    pos_lags = lags[lag > 0:][lag > 0]
    pos_corr = corr[:len(signal)-1]
    
    # Lokalen Peak suchen (nicht Lag 0)
    # Minimum Step-Dauer: 60 BPM = 1 Hz, Maximum: 200 BPM = 3.3 Hz
    min_lag = int(fs / 3.5)   # ~57 samples
    max_lag = int(fs / 0.8)   # ~250 samples
    max_lag = min(max_lag, len(pos_corr))
    
    if max_lag <= min_lag:
        return 0.0  # Zu kurze Signalsektion
    
    # Peak finden
    search_region = pos_corr[min_lag:max_lag]
    peak_idx = np.argmax(search_region)
    peak_lag = min_lag + peak_idx
    peak_val = search_region[peak_idx]
    mean_val = np.mean(pos_corr)
    
    # Signal-to-Noise Ratio des Peaks
    snr = peak_val / mean_val if mean_val > 0 else 0
    
    if snr < 1.5:  # Kein klares periodisches Signal
        return 0.0
    
    step_period_sec = peak_lag / fs
    cadence = 60.0 / step_period_sec
    return cadence


# =========================
# SCHRITT 4: DOMINANTE FREQUENZ via WELSH PERIODOMGRAM
# Kursreferenz: §2.3.2 Welch Periodogram [attached_file:4]
# =========================
def get_dominant_frequency(acc_total, fs):
    """
    Dominante Frequenz via Welch Periodogram.
    Kursreferenz: §2.3.2 — Welch hat niedrigere Varianz als Periodogramm [attached_file:4]
    """
    sig = compute_total_acc(acc_total['x'], acc_total['y'], acc_total['z'])
    sig = bandpass_filter(sig, fs, lowcut=0.5, highcut=5.0)
    
    # Welch: Segment = 2s, Overlap = 50%, Hanning
    freqs, Pxx = welch(sig, fs=fs, nperseg=2*fs, noverlap=fs, window='hann')
    
    # Nur Frequenzband 0.5 – 3.5 Hz betrachten
    mask = (freqs >= 0.5) & (freqs <= 3.5)
    
    if not np.any(mask):
        return 0.0, 0.0
    
    freqs_mask = freqs[mask]
    Pxx_mask = Pxx[mask]
    
    idx_max = np.argmax(Pxx_mask)
    return freqs_mask[idx_max], Pxx_mask[idx_max]


# =========================
# SCHRITT 5: AKTIF-CLASSIFICATION via SCHWELLWERT-LOGIK
# Kursreferenz: §Task3 — nur Signalverarbeitungsmethoden [attached_file:2]
# KEIN Machine Learning für den Klassifikator
# =========================
def classify_activity(cadence, dominant_freq, freq_amplitude):
    """
    Aktivitätsklassifikation über Schwellwert-Logik.
    Basis: Cadence (Schritte/min) + dominante Frequenz (Hz)
    
    Referenzen aus Kursanalyse:
    - Ruhen:   cadence ~ 0, freq ~ 0 Hz, amplitude sehr klein
    - Normalgegehen: cadence ~ 100-120, freq ~ 1-1.5 Hz
    - Schnell Gehen:  cadence ~ 120-150, freq ~ 1.5-2 Hz
    - Rennen:  cadence > 150, freq > 2 Hz
    """
    # Ruhen: sehr niedrige Energie
    if cadence < 30 and dominant_freq < 0.5:
        return 1
    
    # Aktivitäts-Freqeuz-Areas
    if 1.0 <= dominant_freq < 1.6 and 60 <= cadence <= 115:
        return 2  # Normal Gehen
    
    if 1.4 <= dominant_freq < 2.2 and 100 <= cadence <= 160:
        return 3  # Schnell Gehen
    
    if dominant_freq >= 2.0 or cadence > 155:
        return 4  # Rennen
    
    # Grenzfall
    return 0


# =========================
# HAUPTFUNKTION: ALGORITHMUS
# Kursreferenz: §Task3 — Aufnahmetal [attached_file:2]
# =========================
def activity_classifier(acceleration_data, fs):
    """
    Ha这一个csgeneralkation-scale-oriented algorithm for Task 3.
    
    INPUTS:
        - acceleration_data: 3D-Array oder DataFrame mit Acc-(acc_x, acc_y, acc_z)
        - fs: Sampling-Frequenz in Hz (200 für Axiamo X2)
    
    OUTPUTS:
        - cadence: Vector mit Cadence-Schritten pro 5-Sek-Blick
        - activity: Vector mit Integers 0-4 (pro 5-Sek-Blick)
    """
    # ---------- Input-Handling ----------
    if isinstance(acceleration_data, pd.DataFrame):
        # DataFrame mit Spalten x, y, z (von DataProcessor.loadRawDataDevice)
        try:
            acc = {
                'x': acceleration_data['x'].values,
                'y': acceleration_data['y'].values,
                'z': acceleration_data['z'].values
            }
        except KeyError:
            try:
                acc = {
                    'x': acceleration_data['acc_x'].values,
                    'y': acceleration_data['acc_y'].values,
                    'z': acceleration_data['acc_z'].values
                }
            except KeyError:
                print("Fehler: DataFrame hat keine Acc-Spalten (x,y,z)")
                return [], []
    elif isinstance(acceleration_data, np.ndarray):
        # 3D-Array: (N, 3) oder (3, N)
        acc_data = np.array(acceleration_data)
        if acc_data.shape[1] == 3:  # (N, 3)
            acc = {
                'x': acc_data[:, 0],
                'y': acc_data[:, 1],
                'z': acc_data[:, 2]
            }
        elif acc_data.shape[0] == 3:  # (3, N)
            acc = {
                'x': acc_data[0, :],
                'y': acc_data[1, :],
                'z': acc_data[2, :]
            }
        else:
            print("Fehler: Array-Shape nicht unterstützt")
            return [], []
    else:
        print("Fehler: Input-Typ nicht unterstützt")
        return [], []
    
    N = len(acc['x'])
    block_samples = int(BLOCK_DURATION * fs)
    
    cadence_list = []
    activity_list = []
    
    # ---------- Block-Verarbeitung ----------
    num_blocks = N // block_samples
    
    for b in range(num_blocks):
        start = b * block_samples
        end = start + block_samples
        
        block_acc = {
            'x': acc['x'][start:end],
            'y': acc['y'][start:end],
            'z': acc['z'][start:end]
        }
        
        # Kadence via Autokorrelation
        cad = estimate_cadence(block_acc, fs)
        cadence_list.append(cad)
        
        # Dominante Frequenz via Welch Periodogram
        dom_freq, freq_amp = get_dominant_frequency(block_acc, fs)
        
        # Klassifikation via Schwellwert-Logik
        act = classify_activity(cad, dom_freq, freq_amp)
        activity_list.append(act)
    
    return cadence_list, activity_list


# =========================
# TEST MIT DEINEN MESSDATEN
# =========================
if __name__ == "__main__":
    dp = DataProcessor("rawdata/X22/")
    
    # Test mit 3 verschiedenen Aktivitäten
    test_files = [
        ("rawdata/X22/normal_gehen3.pickle",  "Normal Gehen"),
        ("rawdata/X22/schnell_Laufen10.pickle", "Schnell Gehen"),
        ("rawdata/X22/rennen1.pickle", "Rennen")
    ]
    
    for pfad, name in test_files:
        print(f"\n{'='*60}")
        print(f"Test: {name} — Datei: {pfad}")
        print(f"{'='*60}")
        
        dp.loadRawData(pfad)
        device = dp.getDevices()[0]
        dp.loadRawDataDevice(device)
        
        cadences, activities = activity_classifier(dp.dfAcc, FS)
        
        print(f"Anzahl Blöcke: {len(cadences)}")
        print(f"Cadence [Schritte/min]: {[f'{c:.1f}' for c in cadences]}")
        print(f"Aktivität: {[ACTIVITY_MAP[a] for a in activities]}")
        print(f"Mittlere Cadence: {np.mean(cadences):.1f} Schritte/min")